# Ноутбук 06 — Сравнение моделей: почему именно CatBoost?

Хороший ML-инженер не берёт первую попавшуюся модель, а **сравнивает кандидатов честно**: все модели оцениваются на **одних и тех же фолдах**, метрика одна (OOF RMSE). Кто лучше — тот и финал.

План ноутбука:
1. Соберём свой мини-`cross_validate` (как в `experiments.py`).
2. Прогоним бейзлайны: среднее, линейная регрессия, ridge на логитах, сигмоида.
3. Прогоним маленький CatBoost на сырых и на новых признаках.
4. Сравним, сделаем выводы и узнаем главный секрет: **почему при OOF ~10.4 модель берёт 71.21 на лидерборде**.

> Полное сравнение (со сплайнами, LightGBM и нейросетью) делает скрипт `experiments.py` — он дольше, зато полный. Этот ноутбук — быстрая понятная версия.

In [ ]:
# Переходим в корень проекта (если надо) и импортируем всё нужное.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
import pandas as pd  # таблицы
import numpy as np  # математика
import matplotlib.pyplot as plt  # графики
from sklearn.model_selection import KFold  # фолды
from sklearn.linear_model import LinearRegression, RidgeCV  # линейные модели
from scipy.special import expit, logit  # сигмоида и логит
from catboost import CatBoostRegressor  # бустинг
# Наши функции: обработка признаков + старая сигмоидальная модель для сравнения.
from model_utils import generate_features, get_numeric_feature_names, numeric_matrix, fit_preprocessor, transform_features, fit_sigmoid_index, logits_to_score, set_global_seed
set_global_seed(42)  # фиксируем случайность
# Читаем train. data_raw — как есть (52 признака), data_fe — с новыми признаками (61).
data_raw = pd.read_csv("hard_train.csv")
data_fe = generate_features(data_raw)
# Цель y и её логиты z (для моделей, которые учатся на логит-шкале).
y = data_raw["protection_score"].astype(float).to_numpy()
z = logit(np.clip(y / 100.0, 1e-4, 1 - 1e-4))
# Две матрицы: сырая и обогащённая.
raw_matrix = numeric_matrix(data_raw, get_numeric_feature_names(data_raw))
fe_matrix = numeric_matrix(data_fe, get_numeric_feature_names(data_fe))
print("Сырая матрица:", raw_matrix.shape, " Обогащённая:", fe_matrix.shape)

## 1. Наш мини-стенд: cross_validate + rmse

Напишем две маленькие функции: `rmse` считает ошибку, `cross_validate` прогоняет любую модель через 3 фолда (3 вместо 5 — для скорости ноутбука) и возвращает OOF-предсказания.

In [ ]:
# RMSE: корень из среднего квадрата ошибки. Меньше = лучше.
def rmse(prediction, target):
    return float(np.sqrt(np.mean((prediction - target) ** 2)))
# Честный прогон: одна и та же функция для ВСЕХ моделей (одинаковые фолды!).
# fit_predict — функция вида (train_idx, valid_idx) -> предсказания для valid.
def cross_validate(fit_predict, n_rows, n_folds=3, seed=42):
    oof = np.full(n_rows, np.nan)  # массив под OOF-предсказания, пока пустой
    splitter = KFold(n_splits=n_folds, shuffle=True, random_state=seed)  # разбивальщик
    for train_idx, valid_idx in splitter.split(np.arange(n_rows)):  # по всем фолдам
        oof[valid_idx] = fit_predict(train_idx, valid_idx)  # обучить/предсказать, положить на место
    return oof  # вернуть заполненный OOF-массив
# Словарь под итоги: имя модели -> OOF RMSE. Заполним по ходу ноутбука.
results = {}

## 2. Бейзлайн 0: константное среднее

Самая тупая модель: всем предсказать среднее train. Хуже неё быть нельзя — это точка отсчёта.

In [ ]:
# Модель «среднее»: каждому valid — среднее его train-части. np.full размножает число в массив.
oof_mean = cross_validate(lambda tr, va: np.full(len(va), y[tr].mean()), len(y))
results["constant mean"] = rmse(oof_mean, y)  # запоминаем итог
print(f"constant mean OOF RMSE: {results['constant mean']:.4f}")

## 3. Линейная регрессия на сырой цели

Прямая линия (точнее, гиперплоскость): предсказание = w1×признак1 + w2×признак2 + ... + b. Вопрос: линейна ли цель?

In [ ]:
# Линейная регрессия: внутри фолда учим препроцессор, стандартизируем, обучаем, предсказываем.
def linear_fit_predict(tr, va):
    medians, means, scales = fit_preprocessor(raw_matrix[tr])  # параметры — только на train!
    model = LinearRegression().fit(transform_features(raw_matrix[tr], medians, means, scales), y[tr])
    return model.predict(transform_features(raw_matrix[va], medians, means, scales))
oof_linear = cross_validate(linear_fit_predict, len(y))  # прогоняем через стенд
results["linear regression"] = rmse(oof_linear, y)
print(f"linear regression OOF RMSE: {results['linear regression']:.4f}")

## 4. Ridge-регрессия на логитах цели

Идея: цель — это 100 × сигмоида(линейный индекс + шум). Учимся предсказывать логиты (там всё линейно!), а в конце переводим обратно в проценты через сигмоиду.

In [ ]:
# Ridge на логитах: учимся предсказывать z (логиты), возвращаем 100*sigmoid.
# RidgeCV сам выберет силу штрафа alpha из сетки (внутри train, честно).
def ridge_fit_predict(tr, va):
    medians, means, scales = fit_preprocessor(raw_matrix[tr])
    model = RidgeCV(alphas=np.logspace(-3, 3, 13)).fit(transform_features(raw_matrix[tr], medians, means, scales), z[tr])
    return 100.0 * expit(model.predict(transform_features(raw_matrix[va], medians, means, scales)))
oof_ridge = cross_validate(ridge_fit_predict, len(y))
results["ridge on logit"] = rmse(oof_ridge, y)
print(f"ridge on logit OOF RMSE: {results['ridge on logit']:.4f}")

## 5. Сигмоидально-линейная модель (прошлый бейзлайн)

Та же идея, но честнее: минимизируем ошибку СРАЗУ на шкале 0–100 (а не на логитах) методом L-BFGS-B. Раньше это была лучшая модель проекта (~10.16).

In [ ]:
# Сигмоида через fit_sigmoid_index: подбирает веса w и сдвиг b под MSE на шкале 0-100.
def sigmoid_fit_predict(tr, va):
    medians, means, scales = fit_preprocessor(raw_matrix[tr])
    Xt = transform_features(raw_matrix[tr], medians, means, scales)  # train фолда
    Xv = transform_features(raw_matrix[va], medians, means, scales)  # valid фолда
    coefficients, intercept, _ = fit_sigmoid_index(Xt, y[tr])  # w, b (третье не нужно)
    return logits_to_score(Xv @ coefficients + intercept)  # 100*sigmoid(Xv@w + b)
oof_sigmoid = cross_validate(sigmoid_fit_predict, len(y))
results["sigmoid-linear"] = rmse(oof_sigmoid, y)
print(f"sigmoid-linear OOF RMSE: {results['sigmoid-linear']:.4f}")

## 6. Маленький CatBoost: сырые признаки vs новые признаки

А теперь главный кандидат — CatBoost с финальными гиперпараметрами, но маленький (150 деревьев, 3 фолда — для скорости). Прогоним дважды: на сырых 52 признаках и на обогащённых 61 — увидим вклад feature engineering.

In [ ]:
# Фабрика CatBoost-моделей: принимает матрицу, возвращает fit_predict для стенда.
# Гиперпараметры — финальные (глубина 5, сильная регуляризация), итераций меньше (демо!).
def make_catboost(matrix):
    def fit_predict(tr, va):
        medians, means, scales = fit_preprocessor(matrix[tr])
        Xt = transform_features(matrix[tr], medians, means, scales)
        Xv = transform_features(matrix[va], medians, means, scales)
        model = CatBoostRegressor(loss_function="RMSE", eval_metric="RMSE", depth=5, learning_rate=0.03,
                                  l2_leaf_reg=30.0, iterations=150, random_seed=42, verbose=False)
        model.fit(Xt, y[tr], eval_set=(Xv, y[va]), early_stopping_rounds=50, verbose=False)
        return model.predict(Xv)
    return fit_predict
# Прогон 1: сырые признаки.
oof_cb_raw = cross_validate(make_catboost(raw_matrix), len(y))
results["CatBoost (raw)"] = rmse(oof_cb_raw, y)
print(f"CatBoost raw OOF RMSE: {results['CatBoost (raw)']:.4f}")
# Прогон 2: обогащённые признаки (те же настройки!).
oof_cb_fe = cross_validate(make_catboost(fe_matrix), len(y))
results["CatBoost + new features"] = rmse(oof_cb_fe, y)
print(f"CatBoost + new features OOF RMSE: {results['CatBoost + new features']:.4f}")

## 7. Итоговая таблица и график

In [ ]:
# Собираем Series из словаря результатов и сортируем: меньший RMSE (лучше) — сверху.
table = pd.Series(results, name="OOF RMSE (3-fold, demo)").sort_values()
print(table.round(4).to_frame().to_string())  # печатаем как табличку
# Рисуем столбики для наглядности.
plt.figure(figsize=(8, 4))
table[::-1].plot.barh(color="tab:purple")  # переворачиваем: лучшие сверху
plt.xlabel("OOF RMSE (меньше — лучше)")
plt.title("Сравнение моделей на одних фолдах (демо-стенд)")
plt.show()

## 8. Главный секрет: почему OOF ~10.4, а на лидерборде 71.21?

Ты видишь: все модели показывают OOF RMSE около 10–11 на train. А README говорит про **71.21 балла (~RMSE 2.01)** на открытом тесте. Противоречие? Нет! Разгадка:

- **`hard_train.csv` намеренно зашумлён**: к целям добавлен случайный шум (на логит-шкале ~0.6). Его нельзя предсказать в принципе — он случаен. Поэтому OOF на train упёрся в «пол» ~10.2 (см. анализ шума в `reports/eda/eda_summary.md`).
- **`hard_test.csv` чище**: шума в нём меньше, и модель, выучившая настоящий сигнал (а не шум!), показывает там RMSE ~2.

Отсюда стратегия победы:
1. **Сильная регуляризация** (`l2_leaf_reg=30`, неглубокие деревья, ранняя остановка) — не даёт модели выучить шум train.
2. **Хорошие признаки** (feature engineering) — усиливают настоящий сигнал.
3. **Ансамбль** (усреднение 5 фолдов) — гасит случайные ошибки.

Сигмоидально-линейная модель — тоже регуляризованная, но она видит только прямые связи. CatBoost с новыми признаками поймал нелинейности (вроде `cyber_exposure`) и выиграл лидерборд. Полное сравнение с теми же выводами — в `reports/experiments.md` (делается скриптом `experiments.py`).

## Выводы ноутбука 06 (и всего цикла!)

1. Честное сравнение = одни фолды + одна метрика для всех.
2. Лестница качества: среднее (~23.8) → линейная (~11.5) → ridge/sigmoid на логитах (~10.2) → CatBoost с признаками (финал).
3. Train зашумлён специально: OOF ~10.4 — это «пол», а не плохая модель. На чистом тесте — RMSE ~2.
4. Побеждает связка: регуляризация + хорошие признаки + ансамбль.

Ты прошёл весь цикл: данные → EDA → признаки → обучение → предсказание → сравнение. Теперь код проекта (`train.py`, `predict.py`, `model_utils.py`) должен читаться как знакомая история — там те же шаги с построчными комментариями. Удачи на защите! 